In [2]:
import os
import sys
import textwrap

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Configuration

ROOT = os.environ.get("AGL_ROOT", ".")
OUTDIR = os.path.join(ROOT, "Code Outputs", "Figures")

HORIZONS = [1, 3, 6, 12]

LAKES = [
    "Lake Victoria",
    "Lake Tanganyika",
    "Lake Malawi",
    "Lake Kivu",
    "Lake Edward",
    "Lake Albert",
    "Lake Turkana",
]


CANONICAL = {
    "Baseline_metrics.csv": "Code Outputs/Baseline Outputs",
    "FC4_metrics.csv": "Code Outputs/Forecast Var Outputs",
    "FC5_climate_metrics.csv": "Code Outputs/SARIMAX Climate Outputs",
    "FC6_gnn_metrics.csv": "Code Outputs/GNN Outputs",
    "FC7_gnarx_metrics.csv": "Code Outputs/GNARX Outputs",
    "FC7_gnarx_coeffs.csv": "Code Outputs/GNARX Outputs",
    "FC8_null_distribution.csv": "Code Outputs/GNARX Outputs",
    "FC8_sensitivity_summary.csv": "Code Outputs/GNARX Outputs",
    "FC9_gnn_adjacency_summary.csv": "Code Outputs/GNN Outputs",
    "FC9_gnn_null.csv": "Code Outputs/GNN Outputs",
    "FC10_gnn_seed_variance.csv": "Code Outputs/GNN Outputs",
    "FC10_seed_paired.csv": "Code Outputs/GNN Outputs",
    "VARX_diagnostic_metrics.csv": "Code Outputs/VARX Diagnostic Outputs",
}

# -- Figure 5 parameter counts. VERIFIED 11 Aug 2026. -------------------------
# Derived from Forecast Var Fixed.ipynb and confirmed by fitting statsmodels VAR
# on data of the same shape and reading params.shape:
#
#   VAR   exog = SEAS (11 monthly dummies), p = 3 by AIC
#         per equation: 1 const + 3*7 lags + 11 seasonal = 33;  x7 = 231
#   VARX  exog = SEAS (11) + CLIM (7 precip anomalies + DMI = 8), p = 1 by AIC
#         per equation: 1 const + 1*7 lags + 11 seasonal + 8 climate = 27; x7 = 189
#
# VAR/VARX absorb seasonality with 77 dummy coefficients; GNARX removes it in preprocessing with a training-only
# climatology, which costs no estimated parameters
VERIFY_PARAM_COUNTS = False
PARAM_COUNTS = {
    # model : (total, non_seasonal, provenance)
    "VAR": (231, 154, "p=3, exog=11 seasonal dummies; 7 x (1+21+11)"),
    "VARX": (189, 112, "p=1, exog=11 seasonal + 8 climate; 7 x (1+7+11+8)"),
}

# Formatting

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "savefig.dpi": 1000,
        "savefig.bbox": "tight",
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linewidth": 0.6,
        "legend.frameon": False,
        "legend.fontsize": 8,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
    }
)

C = {
    "sarima": "#444444",
    "var": "#7BAFD4",
    "varx": "#1F5F8B",
    "oracle": "#1F5F8B",
    "sarimax": "#4C9A6B",
    "gnn": "#B03A2E",
    "gnar": "#E8A33D",
    "gnarx": "#C1440E",
    "gnarx2": "#8C2F0D",
    "null": "#BBBBBB",
    "hit": "#C1440E",
}

AUDIT = []


def log(msg=""):
    print(msg)
    AUDIT.append(str(msg))


# Loading

_cache = {}


def load(name, required=True):
    if name in _cache:
        return _cache[name]

    candidates = []
    if name in CANONICAL:
        candidates.append(os.path.join(ROOT, CANONICAL[name], name))
    candidates.append(os.path.join(ROOT, name))

    path = next((p for p in candidates if os.path.isfile(p)), None)

    if path is None:  # recursive fallback
        hits = []
        for dirpath, _, files in os.walk(ROOT):
            if name in files:
                hits.append(os.path.join(dirpath, name))
        if len(hits) == 1:
            path = hits[0]
            log(f"  ! {name}: not at canonical path, using {path}")
        elif len(hits) > 1:
            path = sorted(hits)[0]
            log(f"  ! {name}: {len(hits)} copies found, using {path}")
            for h in hits:
                log(f"      candidate: {h}")

    if path is None:
        if required:
            raise FileNotFoundError(
                f"Could not find {name} anywhere under ROOT={os.path.abspath(ROOT)}. "
                f"Set AGL_ROOT to the project root."
            )
        log(f"  ! {name}: NOT FOUND, dependent figure will be skipped")
        _cache[name] = None
        return None

    df = pd.read_csv(path)
    log(f"  loaded {name:32s} {df.shape}  <- {os.path.relpath(path, ROOT)}")
    _cache[name] = df
    return df


def ordinal(v):
    n = int(round(float(v)))
    if 10 <= n % 100 <= 20:
        suf = "th"
    else:
        suf = {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suf}"


def mean_skill(df, model):
    """Mean skill_vs_SARIMA_% across lakes, indexed by horizon."""
    sub = df[df.Model == model]
    if sub.empty:
        return None
    s = sub.groupby("Horizon_m")["skill_vs_SARIMA_%"].mean()
    return s.reindex(HORIZONS)



# Figure 1 - master skill vs horizon


def fig01():
    fc4 = load("FC4_metrics.csv")
    fc5 = load("FC5_climate_metrics.csv")
    fc6 = load("FC6_gnn_metrics.csv")
    fc7 = load("FC7_gnarx_metrics.csv")
    vdx = load("VARX_diagnostic_metrics.csv")

    series = [
        ("VARX", mean_skill(fc4, "VARX"), C["varx"], "-", "o", 2.0, "Joint + climate"),
        ("VAR", mean_skill(fc4, "VAR"), C["var"], "-", "s", 1.4, "Joint + climate"),
        ("SARIMAX", mean_skill(fc5, "SARIMAX"), C["sarimax"], "-", "^", 1.4, "Univariate + climate"),
        ("GNARX_wb", mean_skill(fc7, "GNARX_wb"), C["gnarx"], "-", "D", 2.0, "Parsimonious network"),
        ("GNARX_wb_dmi", mean_skill(fc7, "GNARX_wb_dmi"), C["gnarx2"], "--", "d", 1.3, "Parsimonious network"),
        ("GNAR", mean_skill(fc7, "GNAR"), C["gnar"], "-", "v", 1.3, "Parsimonious network"),
        ("GNN", mean_skill(fc6, "GNN"), C["gnn"], "-", "X", 1.6, "Nonlinear network"),
    ]
    oracle = mean_skill(vdx, "VARX_oracle")

    fig, ax = plt.subplots(figsize=(7.0, 4.4))
    x = np.arange(len(HORIZONS))

    ax.axhline(0, color=C["sarima"], lw=1.4, zorder=2)
    ax.annotate(
        "SARIMA benchmark",
        xy=(x[0], 0),
        xytext=(2, -11),
        textcoords="offset points",
        ha="left",
        va="top",
        fontsize=8,
        color=C["sarima"],
    )

    # Oracle ceiling: legitimate only from h>=3 (see handoff sec.4)
    if oracle is not None:
        mask = np.array(HORIZONS) >= 3
        ax.plot(
            x[mask],
            oracle.values[mask],
            color=C["oracle"],
            ls=":",
            lw=1.4,
            marker="",
            zorder=3,
        )
        ax.annotate(
            "VARX perfect-climate ceiling  (h $\\geq$ 3 only)",
            xy=(x[2], oracle.values[2]),
            xytext=(0, -8),
            textcoords="offset points",
            ha="center",
            va="top",
            fontsize=7.5,
            color=C["oracle"],
            style="italic",
        )

    for name, s, col, ls, mk, lw, _grp in series:
        if s is None:
            log(f"  ! fig01: {name} missing, skipped")
            continue
        ax.plot(x, s.values, color=col, ls=ls, lw=lw, marker=mk, ms=4.5, label=name, zorder=4)
        log(f"  fig01  {name:14s} " + "  ".join(f"h{h}={v:6.1f}" for h, v in zip(HORIZONS, s.values)))

    ax.set_xticks(x)
    ax.set_xticklabels([f"{h}" for h in HORIZONS])
    ax.set_xlabel("Forecast horizon (months)")
    ax.set_ylabel("Mean skill vs SARIMA (%)")
    ax.set_title(
        "Forecast skill relative to the univariate SARIMA benchmark\n"
        "positive = beats SARIMA; mean over seven lakes",
        loc="left",
    )
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=4)
    fig.text(
        0.5,
        -0.145,
        "Random walk (-53.4 / -65.8 / -47.2 / -0.8) omitted for scale. "
        "The perfect-climate ceiling is shown from h=3 because at h=1 it is provably identical to the honest VARX.",
        ha="center",
        fontsize=7.5,
        style="italic",
        color="#666666",
    )

    save(fig, "FIG_01_master_skill.svg")



# Figure 2 - model x lake heatmap at h=1



def fig02(horizon=1):
    fc4 = load("FC4_metrics.csv")
    fc5 = load("FC5_climate_metrics.csv")
    fc6 = load("FC6_gnn_metrics.csv")
    fc7 = load("FC7_gnarx_metrics.csv")

    spec = [
        ("VAR", fc4),
        ("VARX", fc4),
        ("SARIMAX", fc5),
        ("GNAR", fc7),
        ("GNARX_wb", fc7),
        ("GNARX_wb_dmi", fc7),
        ("GNN", fc6),
    ]

    rows, labels = [], []
    for model, df in spec:
        sub = df[(df.Model == model) & (df.Horizon_m == horizon)]
        if sub.empty:
            log(f"  ! fig02: {model} missing at h={horizon}, skipped")
            continue
        s = sub.set_index("Lake")["skill_vs_SARIMA_%"].reindex(LAKES)
        rows.append(s.values)
        labels.append(model)

    M = np.array(rows, dtype=float)
    lim = np.nanmax(np.abs(M))

    fig, ax = plt.subplots(figsize=(7.4, 3.5))
    im = ax.imshow(M, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
    im.set_cmap("RdBu")

    ax.set_xticks(np.arange(len(LAKES)))
    ax.set_xticklabels([l.replace("Lake ", "") for l in LAKES], rotation=30, ha="right")
    ax.set_yticks(np.arange(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_xticks(np.arange(-0.5, len(LAKES), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(labels), 1), minor=True)
    ax.grid(which="minor", color="white", lw=1.2)
    ax.grid(which="major", visible=False)
    ax.tick_params(which="minor", length=0)

    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            ax.text(
                j,
                i,
                f"{v:.1f}",
                ha="center",
                va="center",
                fontsize=7.5,
                color="white" if abs(v) > 0.55 * lim else "#222222",
            )

    cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cb.set_label("skill vs SARIMA (%)", fontsize=8)
    cb.ax.tick_params(labelsize=7)

    ax.set_title(
        f"Per-lake forecast skill at h={horizon}: the regional gain is not uniform\n"
        "blue = beats SARIMA, red = worse",
        loc="left",
    )
    log(f"  fig02  matrix {M.shape}, range [{np.nanmin(M):.1f}, {np.nanmax(M):.1f}]")
    for lab, r in zip(labels, M):
        log(f"    {lab:14s} " + "  ".join(f"{l.replace('Lake ',''):>11s}={v:6.1f}" for l, v in zip(LAKES, r)))

    save(fig, "FIG_02_lake_model_heatmap.svg")



# Figure 3 - GNARX beta attenuation



def fig03():
    co = load("FC7_gnarx_coeffs.csv")

    order = ["GNAR", "GNARX_wb", "GNARX_wb_dmi"]
    pretty = ["GNAR", "GNARX$_{wb}$", "GNARX$_{wb,dmi}$"]

    def grab(model, term):
        r = co[(co.Model == model) & (co.term == term)]
        if r.empty:
            return None
        r = r.iloc[0]
        return dict(
            coef=float(r.coef),
            se=float(r.se_clustered_by_time),
            z=float(r.z_clustered),
            p=float(r.p_value_clustered),
        )

    beta = [grab(m, "beta_lag1_stage1") for m in order]
    # exogenous terms: exog0 = water balance, exog1 = DMI
    lam_wb = [grab(m, "lambda_exog0_lag1") for m in order]
    lam_dmi = [grab(m, "lambda_exog1_lag1") for m in order]

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(7.4, 3.9), gridspec_kw={"width_ratios": [1, 1]})
    x = np.arange(3)

    # left: the network coefficient collapsing
    axL.axhline(0, color="#888888", lw=1.0)
    for i, b in enumerate(beta):
        if b is None:
            continue
        lo, hi = b["coef"] - 1.96 * b["se"], b["coef"] + 1.96 * b["se"]
        sig = b["p"] < 0.05
        col = C["gnarx"] if sig else "#999999"
        axL.plot([i, i], [lo, hi], color=col, lw=2.0, solid_capstyle="round", zorder=3)
        axL.plot([i], [b["coef"]], "o", color=col, ms=7, zorder=4,
                 markerfacecolor=col if sig else "white", markeredgewidth=1.6)
        axL.annotate(
            f"z = {b['z']:.2f}\np = {b['p']:.4f}" if b["p"] >= 1e-4 else f"z = {b['z']:.2f}\np < 0.0001",
            xy=(i, hi),
            xytext=(0, 7),
            textcoords="offset points",
            ha="center",
            fontsize=7.5,
            color=col,
        )
        log(f"  fig03  beta {order[i]:14s} coef={b['coef']:.4f}  se={b['se']:.4f}  z={b['z']:.3f}  p={b['p']}")

    axL.set_xticks(x)
    axL.set_xticklabels(pretty, fontsize=8.5)
    axL.set_xlim(-0.55, 2.55)
    axL.set_ylabel(r"$\beta_{1}$  (network / neighbour term)")
    axL.set_title("(a) the network coefficient collapses", loc="left", fontsize=9)
    axL.margins(y=0.30)

    # right: the climate coefficients steady
    axR.axhline(0, color="#888888", lw=1.0)
    plotted = False
    for i in range(3):
        for d, col, off, lab in (
            (lam_wb[i], C["sarimax"], -0.10, r"$\lambda_{wb}$"),
            (lam_dmi[i], "#3E6B8C", 0.10, r"$\lambda_{dmi}$"),
        ):
            if d is None:
                continue
            plotted = True
            lo, hi = d["coef"] - 1.96 * d["se"], d["coef"] + 1.96 * d["se"]
            axR.plot([i + off, i + off], [lo, hi], color=col, lw=2.0, solid_capstyle="round", zorder=3)
            axR.plot([i + off], [d["coef"]], "o", color=col, ms=6.5, zorder=4)
            axR.annotate(f"z={d['z']:.1f}", xy=(i + off, hi), xytext=(0, 5),
                         textcoords="offset points", ha="center", fontsize=7, color=col)
            log(f"  fig03  {lab:12s} {order[i]:14s} coef={d['coef']:.4f}  se={d['se']:.4f}  z={d['z']:.3f}")

    axR.set_xticks(x)
    axR.set_xticklabels(pretty, fontsize=8.5)
    axR.set_xlim(-0.55, 2.55)
    axR.set_ylabel("exogenous climate coefficient")
    axR.set_title("(b) the climate coefficients do not move", loc="left", fontsize=9)
    axR.margins(y=0.30)
    if plotted:
        axR.legend(
            handles=[
                Line2D([], [], color=C["sarimax"], marker="o", lw=2, label=r"$\lambda_{wb}$  water balance"),
                Line2D([], [], color="#3E6B8C", marker="o", lw=2, label=r"$\lambda_{dmi}$  dipole mode index"),
            ],
            loc="lower left",
        )

    fig.suptitle(
        "Coefficient-level evidence that the cross-lake network carries climate, not extra information",
        fontsize=10, x=0.5, y=1.02,
    )
    fig.tight_layout()
    fig.text(0.5, -0.06,
             "Ablations left to right add climate covariates. Bars are 95% intervals from standard errors clustered by date\n"
             "(the seven lakes are contemporaneously correlated). Hollow marker = not significant at the 5% level.",
             ha="center", fontsize=7.5, style="italic", color="#666666")

    save(fig, "FIG_03_gnarx_beta_attenuation.svg")



# Figure 4 - the two null controls, side by side



def fig04(hs=(1, 6)):
    n8 = load("FC8_null_distribution.csv")
    s8 = load("FC8_sensitivity_summary.csv")
    n9 = load("FC9_gnn_null.csv")
    s9 = load("FC9_gnn_adjacency_summary.csv")

    n8 = n8[n8.Model == "GNARX_wb"]
    real8 = s8[s8.Model == "GNARX_wb"].set_index("Graph")
    real9 = s9.set_index("Graph")

    fig, axes = plt.subplots(len(hs), 2, figsize=(7.6, 2.5 * len(hs) + 0.8))
    if len(hs) == 1:
        axes = np.array([axes])

    for r, h in enumerate(hs):
        col = f"h{h}"

        # ---- left: FC8, linear GNARX
        ax = axes[r, 0]
        vals = n8[col].values
        ax.hist(vals, bins=18, color=C["null"], edgecolor="white", lw=0.6)
        for g in real8.index:
            v = real8.loc[g, col]
            prim = g == "ccm_diff_train"
            ax.axvline(v, color=C["hit"] if prim else "#666666",
                       lw=2.0 if prim else 0.9, ls="-" if prim else "--", zorder=5)
        pct = real8.loc["ccm_diff_train", f"null_pctile_{col}"]
        v = real8.loc["ccm_diff_train", col]
        ax.annotate(f"CCM graph\n{v:.1f}%  ({ordinal(pct)} pctile)",
                    xy=(v, ax.get_ylim()[1] * 0.92), xytext=(6, 0), textcoords="offset points",
                    fontsize=7.5, color=C["hit"], va="top")
        ax.set_title(f"GNARX (linear), h={h}", loc="left")
        ax.set_xlabel("mean skill vs SARIMA (%)")
        ax.set_ylabel(f"random graphs (n={len(vals)})")
        log(f"  fig04  FC8 h={h}: null mean {vals.mean():.2f} [{np.percentile(vals,5):.2f},"
            f" {np.percentile(vals,95):.2f}], CCM {v:.1f} at pctile {pct}")

        # ---- right: FC9, nonlinear GNN
        ax = axes[r, 1]
        vals9 = n9[col].values
        ax.hist(vals9, bins=12, color=C["null"], edgecolor="white", lw=0.6)
        for g in real9.index:
            v = real9.loc[g, col]
            key = g in ("identity", "corr_abs_asbuilt")
            ax.axvline(v, color=C["hit"] if key else "#666666",
                       lw=2.0 if key else 0.9, ls="-" if key else "--", zorder=5)
        top = ax.get_ylim()[1]
        for g, dy in (("identity", 0.94), ("corr_abs_asbuilt", 0.70)):
            v = real9.loc[g, col]
            lbl = "no graph" if g == "identity" else "published graph"
            ax.annotate(f"{lbl}\n{v:.1f}%", xy=(v, top * dy), xytext=(-6, 0),
                        textcoords="offset points", fontsize=7.5, color=C["hit"],
                        va="top", ha="right")
        ax.set_title(f"GNN (nonlinear), h={h}", loc="left")
        ax.set_xlabel("mean skill vs SARIMA (%)")
        ax.set_ylabel(f"random graphs (n={len(vals9)})")
        log(f"  fig04  FC9 h={h}: null mean {vals9.mean():.2f} [min {vals9.min():.2f},"
            f" max {vals9.max():.2f}], identity {real9.loc['identity', col]:.1f},"
            f" asbuilt {real9.loc['corr_abs_asbuilt', col]:.1f}")

    fig.suptitle(
        "Two calibrated null controls. Left: the CCM graph is indistinguishable from a random\n"
        "density-matched graph. Right: structured graphs beat random ones, but no graph beats no graph.",
        fontsize=9.5, x=0.01, ha="left",
    )
    fig.text(0.5, -0.03,
             "Grey bars = density-matched random directed graphs. "
             "Dashed lines = the other real graphs tested.",
             ha="center", fontsize=7.5, style="italic", color="#666666")
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    save(fig, "FIG_04_null_controls.svg", tight=False)



# Figure 5 - parsimony



def fig05(horizon=1):
    fc4 = load("FC4_metrics.csv")
    fc7 = load("FC7_gnarx_metrics.csv")
    co = load("FC7_gnarx_coeffs.csv")

    npar = co.groupby("Model")["n_params"].first().to_dict()

    # (name, total params, non-seasonal params, skill, colour)
    pts = []
    for m in ("GNAR", "GNARX_wb", "GNARX_wb_dmi"):
        s = mean_skill(fc7, m)
        if s is not None and m in npar:
            n = int(npar[m])
            pts.append((m, n, n, float(s.loc[horizon]), C["gnarx"], True))
    for m in ("VAR", "VARX"):
        s = mean_skill(fc4, m)
        tot, nonseas, _prov = PARAM_COUNTS.get(m, (None, None, ""))
        if s is not None and tot is not None:
            pts.append((m, tot, nonseas, float(s.loc[horizon]), C["varx"], True))

    # paired horizontal bars: avoids the label collisions of a scatter, and the
    # log parameter axis and the linear skill axis stay separate and readable
    pts.sort(key=lambda t: t[3])
    names = [p[0] for p in pts]
    y = np.arange(len(pts))

    fig, (axS, axP) = plt.subplots(
        1, 2, figsize=(7.8, 3.4), sharey=True, gridspec_kw={"width_ratios": [1.25, 1]}
    )

    for i, (name, tot, nonseas, sk, col, _v) in enumerate(pts):
        axS.barh(i, sk, color=col, edgecolor=col, height=0.62)
        axS.annotate(f"{sk:+.1f}%", xy=(sk, i), xytext=(4 if sk >= 0 else -4, 0),
                     textcoords="offset points", va="center",
                     ha="left" if sk >= 0 else "right", fontsize=8)
        # full bar = every estimated coefficient; inner bar = excluding the
        # seasonal dummies, which GNARX does not need because it deseasonalises
        axP.barh(i, tot, color=col, edgecolor=col, height=0.62, alpha=0.35)
        axP.barh(i, nonseas, color=col, edgecolor=col, height=0.62)
        lbl = f"{tot}" if tot == nonseas else f"{tot}   ({nonseas} non-seasonal)"
        axP.annotate(lbl, xy=(tot, i), xytext=(4, 0), textcoords="offset points",
                     va="center", ha="left", fontsize=8)
        log(f"  fig05  {name:14s} params={tot:4d} (non-seasonal {nonseas:4d})  "
            f"h{horizon} skill={sk:+.1f}")

    axS.axvline(0, color=C["sarima"], lw=1.2)
    axS.set_yticks(y)
    axS.set_yticklabels(names)
    axS.set_xlabel(f"mean skill vs SARIMA (%) at h={horizon}")
    axS.set_title("(a) what it buys you", loc="left", fontsize=9)
    axS.margins(x=0.18)
    axS.grid(axis="y", visible=False)

    axP.set_xscale("log")
    axP.set_xlim(5, 900)
    axP.set_xlabel("estimated coefficients (log scale)")
    axP.set_title("(b) what it costs you", loc="left", fontsize=9)
    axP.grid(axis="y", visible=False)
    axP.legend(handles=[
        Line2D([], [], color="#777777", lw=7, alpha=0.35, label="incl. seasonal dummies"),
        Line2D([], [], color="#777777", lw=7, label="excl. seasonal dummies"),
    ], loc="center right", fontsize=7)

    fig.suptitle(
        "Parsimony: GNARX recovers 72% of the VARX gain from 9 coefficients instead of 189",
        fontsize=10, x=0.5, y=1.03,
    )
    fig.tight_layout()
    fig.text(0.5, -0.12,
             "GNAR/GNARX counts are read from FC7_gnarx_coeffs.csv; VAR and VARX counts are derived from Forecast Var Fixed.ipynb\n"
             "(VAR p=3, VARX p=1, both with 11 monthly dummies) and confirmed against the fitted parameter matrix. VAR and VARX\n"
             "absorb seasonality with 77 dummy coefficients; GNARX removes it in preprocessing at no parameter cost.",
             ha="center", fontsize=7.5, style="italic", color="#666666")

    save(fig, "FIG_05_parsimony.svg")



# Figure 6 - GNN seed variance (FC10)



def fig06():
    V = load("FC10_gnn_seed_variance.csv", required=False)
    PR = load("FC10_seed_paired.csv", required=False)
    if V is None or PR is None:
        log("  ! fig06: FC10 files not found, skipped")
        return

    W = (V.groupby(["Graph", "replicate", "Horizon_m"])["skill_vs_SARIMA_%"]
         .mean().unstack())
    W.columns = [f"h{c}" for c in W.columns]
    W = W.reset_index()

    order = ["identity", "ccm_symmetric", "corr_abs_asbuilt", "ccm_directed",
             "ccm_symmetric_row"]
    order = [g for g in order if g in set(W.Graph)]
    pretty = {"identity": "identity\n(no graph)", "ccm_symmetric": "ccm_symmetric",
              "corr_abs_asbuilt": "corr_abs_asbuilt\n(published FC6)",
              "ccm_directed": "ccm_directed",
              "ccm_symmetric_row": "ccm_symmetric_row"}

    fig, axes = plt.subplots(1, 2, figsize=(7.8, 4.0),
                             gridspec_kw={"width_ratios": [1.35, 1]})

    # (a) spread across replicates, with the FC9 replicate marked
    ax = axes[0]
    for i, g in enumerate(order):
        v = W[W.Graph == g]["h1"].values
        fc9 = W[(W.Graph == g) & (W.replicate == 0)]["h1"].iloc[0]
        col = C["gnarx"] if g == "identity" else "#5A5A5A"
        ax.plot([v.min(), v.max()], [i, i], color=col, lw=2.0, alpha=0.45,
                solid_capstyle="round", zorder=2)
        ax.plot(v, np.full(len(v), i), "o", ms=4, color=col, alpha=0.75, zorder=3)
        ax.plot([v.mean()], [i], "|", ms=18, mew=2.2, color=col, zorder=4)
        ax.plot([fc9], [i], "*", ms=13, color=C["varx"], zorder=5)
        log(f"  fig06  {g:20s} h1 mean {v.mean():7.2f}  sd {v.std(ddof=1):5.2f}  "
            f"[{v.min():6.2f}, {v.max():6.2f}]  FC9 replicate {fc9:7.2f}")

    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([pretty[g] for g in order], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("mean skill vs SARIMA (%) at h=1")
    ax.set_title("(a) each dot is one 3-seed replicate", loc="left", fontsize=9)
    ax.grid(axis="y", visible=False)
    ax.legend(handles=[
        Line2D([], [], ls="", marker="*", ms=11, color=C["varx"],
               label="the replicate FC9 and FC6 reported"),
        Line2D([], [], ls="", marker="|", ms=12, mew=2, color="#5A5A5A",
               label="mean of 10 replicates"),
    ], loc="lower right", fontsize=7)

    # (b) paired differences vs identity
    ax = axes[1]
    hs = [1, 3, 6, 12]
    graphs_b = [g for g in order if g != "identity"]
    marks = ["o", "s", "^", "D"]
    for k, g in enumerate(graphs_b):
        sub = PR[PR.graph == g].set_index("Horizon_m")
        xs = [sub.loc[h, "mean_diff"] for h in hs if h in sub.index]
        los = [sub.loc[h, "ci_lo"] for h in hs if h in sub.index]
        his = [sub.loc[h, "ci_hi"] for h in hs if h in sub.index]
        yy = np.arange(len(xs)) + (k - 1.5) * 0.16
        for x, lo, hi, yv in zip(xs, los, his, yy):
            sig = lo > 0 or hi < 0
            ax.plot([lo, hi], [yv, yv], color=C["gnarx"] if sig else "#AAAAAA", lw=1.6)
            ax.plot([x], [yv], marks[k], ms=4.5,
                    color=C["gnarx"] if sig else "#AAAAAA",
                    markerfacecolor=(C["gnarx"] if sig else "white"))
    ax.axvline(0, color=C["sarima"], lw=1.2)
    ax.set_yticks(np.arange(len(hs)))
    ax.set_yticklabels([f"h={h}" for h in hs])
    ax.invert_yaxis()
    ax.set_xlabel("identity minus graph (% points)")
    ax.set_title("(b) paired, 95% CI", loc="left", fontsize=9)
    ax.grid(axis="y", visible=False)
    ax.legend(handles=[Line2D([], [], ls="", marker=marks[k], ms=5,
                              color="#5A5A5A", label=g)
                       for k, g in enumerate(graphs_b)],
              loc="lower right", fontsize=6.5)

    fig.suptitle(
        "Seed variability: the published GNN result is the best of ten replicates",
        fontsize=10, x=0.5, y=1.02,
    )
    fig.tight_layout()
    fig.text(0.5, -0.07,
             "Ten independent 3-seed replicates per graph, seeds (0,1,2) ... (27,28,29); replicate 0 is the one FC6 and FC9 used.\n"
             "Panel (b) differences are paired within replicate. Filled markers mark intervals excluding zero.",
             ha="center", fontsize=7.5, style="italic", color="#666666")

    save(fig, "FIG_06_gnn_seed_variance.svg")







def save(fig, name, tight=True):
    os.makedirs(OUTDIR, exist_ok=True)
    path = os.path.join(OUTDIR, name)
    fig.savefig(path, bbox_inches="tight" if tight else None)
    plt.close(fig)
    log(f"  -> wrote {name}")


def main():
    log("=" * 78)
    log("DISSERTATION FIGURES")
    log("=" * 78)
    log(f"  ROOT   = {os.path.abspath(ROOT)}")
    log(f"  OUTDIR = {os.path.abspath(OUTDIR)}")
    log("")
    log("-- loading -------------------------------------------------------------")

    figs = [
        ("FIG 01  master skill vs horizon", fig01),
        ("FIG 02  per-lake heatmap", fig02),
        ("FIG 03  GNARX beta attenuation", fig03),
        ("FIG 04  null controls", fig04),
        ("FIG 05  parsimony", fig05),
        ("FIG 06  GNN seed variance", fig06),
    ]

    failures = []
    for title, fn in figs:
        log("")
        log(f"-- {title} " + "-" * max(0, 68 - len(title)))
        try:
            fn()
        except Exception as exc:  # keep going, report at the end
            failures.append((title, repr(exc)))
            log(f"  !! FAILED: {exc!r}")

    log("")
    log("=" * 78)
    if VERIFY_PARAM_COUNTS:
        log("WARNING - Figure 5 parameter counts that are NOT read from any file:")
        for k, (n, prov) in PARAM_COUNTS.items():
            log(f"   {k:10s} n={n}   {prov}")
        log("   Confirm these against Forecast Var Fixed.ipynb before using Figure 5.")
        log("")
    if failures:
        log(f"{len(failures)} FIGURE(S) FAILED:")
        for t, e in failures:
            log(f"   {t}: {e}")
    else:
        log(f"All {len(figs)} figures written successfully.")
    log("=" * 78)

    os.makedirs(OUTDIR, exist_ok=True)
    with open(os.path.join(OUTDIR, "FIG_data_audit.txt"), "w") as fh:
        fh.write("\n".join(AUDIT) + "\n")
    print(f"\nAudit trail -> {os.path.join(OUTDIR, 'FIG_data_audit.txt')}")

    return 1 if failures else 0


if __name__ == "__main__":
    (main())

DISSERTATION FIGURES
  ROOT   = /Users/russellsharp/MSc/ERP/Repository/Repo
  OUTDIR = /Users/russellsharp/MSc/ERP/Repository/Repo/Code Outputs/Figures

-- loading -------------------------------------------------------------

-- FIG 01  master skill vs horizon -------------------------------------
  loaded FC4_metrics.csv                  (112, 6)  <- Code Outputs/Forecast Var Outputs/FC4_metrics.csv
  loaded FC5_climate_metrics.csv          (84, 7)  <- Code Outputs/SARIMAX Climate Outputs/FC5_climate_metrics.csv
  loaded FC6_gnn_metrics.csv              (140, 6)  <- Code Outputs/GNN Outputs/FC6_gnn_metrics.csv
  loaded FC7_gnarx_metrics.csv            (224, 7)  <- Code Outputs/GNARX Outputs/FC7_gnarx_metrics.csv
  loaded VARX_diagnostic_metrics.csv      (196, 6)  <- Code Outputs/VARX Diagnostic Outputs/VARX_diagnostic_metrics.csv
  fig01  VARX           h1=  11.7  h3=   6.0  h6=   4.1  h12=   3.0
  fig01  VAR            h1=   0.5  h3=   2.7  h6=   3.3  h12=   0.3
  fig01  SARIMAX    